# TL8 — Árboles, bagging, bosques y boosting

## Índice

1. [Introducción](#1-introducción)
2. [Clasificación](#2-clasificación)
3. [Regresión](#3-regresión)
4. [Ejercicio de clasificación](#4-ejercicio-de-clasificación)
5. [Ejercicio de regresión](#5-ejercicio-de-regresión)

> **Cita:** Juan Ciscar, Alfons. *"Material docente de la asignatura Aprendizaje Automático (APA)"*. Grado de Inteligencia Artificial (GIAR), Escuela Técnica Superior de Ingeniería Informática (ETSINF), Dep. de Sistemas Informáticos y Computación (DSIC), Universitat Politècnica de València (UPV), primer semestre del curso 2026/27.

---
## 1. Introducción

- **Classification and regression trees (CART):** árboles para clasificación y regresión aprendidos con métodos voraces.
- **Bagging:** ensamble de modelos ajustados con diferentes versiones de los datos, obtenidas por *bootstrapping*.
- **Bosques:** variante de bagging que también aleatoriza variables de entrada.
- **Boosting:** ensamble de modelos ajustado por etapas hacia adelante o *forward stagewise additive modeling* (FSAM).
- **Gradient boosting:** visión actual de boosting.

**Objetivo:** aplicar CART, bagging, bosques y (gradient) boosting a tareas de clasificación y regresión.

**Limitación computacional:** experimentos diseñados para terminar en pocos minutos de computación.

---
## 2. Clasificación

### 2.0 MNIST: lectura y reducción de la dimensión ($K = 400$)

Salida esperada: `(60000, 400) (60000,) (10000, 400) (10000,)`

In [ ]:
import numpy as np
import datasets
from sklearn.decomposition import PCA

ds = datasets.load_dataset("ylecun/mnist").with_format("numpy")
X_train = ds['train'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)
X_test = ds['test'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)

K = 400
pca = PCA(n_components=K)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

### 2.1 CART: `DecisionTreeClassifier`

| Parámetro | Descripción |
|-----------|-------------|
| `max_depth=None` | profundidad máxima, ilimitada por omisión |

Salida esperada:

```text
Precisión: 83.6% con {'max_depth': 20}
Tiempo (hh:mm:ss): 00:01:12
```

In [ ]:
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

clf = DecisionTreeClassifier(random_state=23)
G = {"max_depth": [2, 5, 10, 20, 50]}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
GS = GridSearchCV(clf, G, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=8)
acc = GS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.1%} con {GS.best_params_}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 2.2 Bagging: `BaggingClassifier`

| Parámetro | Descripción |
|-----------|-------------|
| `estimator=None` | usamos `DecisionTreeClassifier()` con `estimator__max_depth` a ajustar |
| `n_estimators=10` | número de estimadores del ensamble |

Salida esperada:

```text
Precisión: 87.8% con {'estimator__max_depth': 20, 'n_estimators': 5}
Tiempo (hh:mm:ss): 00:02:36
```

In [ ]:
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

clf = BaggingClassifier(estimator=DecisionTreeClassifier(), random_state=23)
G = {"estimator__max_depth": [20], "n_estimators": [5]}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
GS = GridSearchCV(clf, G, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=8)
acc = GS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.1%} con {GS.best_params_}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 2.3 Bosques: `RandomForestClassifier`

| Parámetro | Descripción |
|-----------|-------------|
| `n_estimators=100` | número de estimadores del ensamble |
| `max_depth=None` | profundidad máxima, ilimitada por omisión |

Salida esperada:

```text
Precisión: 93.6% con {'max_depth': 20, 'n_estimators': 100}
Tiempo (hh:mm:ss): 00:02:22
```

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

clf = RandomForestClassifier(random_state=23)
G = {"n_estimators": [100], "max_depth": [20]}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
GS = GridSearchCV(clf, G, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=8)
acc = GS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.1%} con {GS.best_params_}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 2.4 AdaBoost: `AdaBoostClassifier`

| Parámetro | Descripción |
|-----------|-------------|
| `estimator=None` | usamos `DecisionTreeClassifier()` con `estimator__max_depth` a ajustar |
| `n_estimators=50` | número de estimadores del ensamble |

Salida esperada:

```text
Precisión: 84.0% con {'estimator__max_depth': 20, 'n_estimators': 3}
Tiempo (hh:mm:ss): 00:02:24
```

In [ ]:
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

clf = AdaBoostClassifier(estimator=DecisionTreeClassifier(), random_state=23)
G = {"estimator__max_depth": [20], "n_estimators": [3]}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
GS = GridSearchCV(clf, G, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=8)
acc = GS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.1%} con {GS.best_params_}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 2.5 Gradient boosting: `GradientBoostingClassifier`

| Parámetro | Descripción |
|-----------|-------------|
| `n_estimators=100` | número de estimadores del ensamble |
| `max_depth=3` | profundidad máxima |

Salida esperada:

```text
Precisión: 72.0% con {'max_depth': 3, 'n_estimators': 2}
Tiempo (hh:mm:ss): 00:03:10
```

In [ ]:
import time
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

clf = GradientBoostingClassifier(random_state=23)
G = {"n_estimators": [2], "max_depth": [3]}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
GS = GridSearchCV(clf, G, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=8)
acc = GS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.1%} con {GS.best_params_}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 2.6 Gradient boosting para datasets grandes (10K+ datos): `HistGradientBoostingClassifier`

| Parámetro | Descripción |
|-----------|-------------|
| `max_depth=None` | profundidad máxima, ilimitada por omisión |

Salida esperada:

```text
Precisión: 96.3% con {'max_depth': 20}
Tiempo (hh:mm:ss): 00:01:50
```

In [ ]:
import time
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

clf = HistGradientBoostingClassifier(random_state=23)
G = {"max_depth": [20]}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
GS = GridSearchCV(clf, G, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=8)
acc = GS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.1%} con {GS.best_params_}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 2.7 Gradient boosting con XGBoost: `XGBClassifier`

| Parámetro | Descripción |
|-----------|-------------|
| `max_depth=None` | profundidad máxima, ilimitada por omisión |

Salida esperada:

```text
Precisión: 95.6% con {'max_depth': 4}
Tiempo (hh:mm:ss): 00:02:28
```

In [ ]:
import time
from xgboost import XGBClassifier
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

clf = XGBClassifier(tree_method="hist", random_state=23)
G = {"max_depth": [4]}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
GS = GridSearchCV(clf, G, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=8)
acc = GS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.1%} con {GS.best_params_}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

---
## 3. Regresión

### 3.0 auto-mpg: carga y preproceso

**auto-mpg:** predicción de consumo en *miles per gallon*. Además de la versión estandarizada se prepara una con características polinómicas de grado 2 (`X_train_poly`, `X_test_poly`).

In [ ]:
import numpy as np
import datasets
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

ds = datasets.load_dataset("scikit-learn/auto-mpg")
df = ds['train'].to_pandas()
df.drop(df[df.horsepower == "?"].index, inplace=True)
df = df.astype({'horsepower' : np.int64})
df['origin'] = df['origin'].map({1: 'USA', 2: 'Europe', 3: 'Japan'})
df = df.astype({'origin' : 'category'})
df.drop('car name', axis=1, inplace=True)
df = pd.concat([df, pd.get_dummies(df.origin, prefix='', prefix_sep='')], axis=1)
df.drop('origin', axis=1, inplace=True)
y = df.pop('mpg')
X = df

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

### 3.1 CART: `DecisionTreeRegressor`, `max_depth=None`

Salida esperada:

```text
R2 (sin): 81.7% con {'max_depth': 5}
R2 (con): 81.0% con {'max_depth': 5}
Tiempo (hh:mm:ss): 00:00:02
```

In [ ]:
import time
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

rgs = DecisionTreeRegressor(random_state=23)
G = {"max_depth": [2, 5, 10, 20, 50, None]}
splitter = ShuffleSplit(n_splits=10, test_size=0.1, random_state=23)
GS = GridSearchCV(rgs, G, scoring='r2', n_jobs=-1, cv=splitter, pre_dispatch=8)
r2 = GS.fit(X_train, y_train).score(X_test, y_test)
print(f"R2 (sin): {r2:.1%} con {GS.best_params_}")

GS_poly = GS
r2_poly = GS_poly.fit(X_train_poly, y_train).score(X_test_poly, y_test)
print(f"R2 (con): {r2_poly:.1%} con {GS_poly.best_params_}")

print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 3.2 Bagging: `BaggingRegressor`, `estimator=None`, `n_estimators=10`

Salida esperada:

```text
R2 (sin): 88.0% con {'estimator__max_depth': 20, 'n_estimators': 500}
R2 (con): 88.5% con {'estimator__max_depth': 20, 'n_estimators': 500}
Tiempo (hh:mm:ss): 00:00:20
```

In [ ]:
import time
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

rgs = BaggingRegressor(estimator=DecisionTreeRegressor(), random_state=23)
G = {"estimator__max_depth": [2, 5, 10, 20, 50, None], "n_estimators": [10, 20, 50, 100, 200, 500]}
splitter = ShuffleSplit(n_splits=10, test_size=0.1, random_state=23)
GS = GridSearchCV(rgs, G, scoring='r2', n_jobs=-1, cv=splitter, pre_dispatch=8)
r2 = GS.fit(X_train, y_train).score(X_test, y_test)
print(f"R2 (sin): {r2:.1%} con {GS.best_params_}")

GS_poly = GS
r2_poly = GS_poly.fit(X_train_poly, y_train).score(X_test_poly, y_test)
print(f"R2 (con): {r2_poly:.1%} con {GS_poly.best_params_}")

print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 3.3 Bosques: `RandomForestRegressor`, `n_estimators=100`, `max_depth=None`

Salida esperada:

```text
R2 (sin): 88.2% con {'max_depth': 10, 'n_estimators': 500}
R2 (con): 88.4% con {'max_depth': 10, 'n_estimators': 500}
Tiempo (hh:mm:ss): 00:00:17
```

In [ ]:
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

rgs = RandomForestRegressor(random_state=23)
G = {"n_estimators": [10, 20, 50, 100, 200, 500], "max_depth": [2, 5, 10, 20, 50, None]}
splitter = ShuffleSplit(n_splits=10, test_size=0.1, random_state=23)
GS = GridSearchCV(rgs, G, scoring='r2', n_jobs=-1, cv=splitter, pre_dispatch=8)
r2 = GS.fit(X_train, y_train).score(X_test, y_test)
print(f"R2 (sin): {r2:.1%} con {GS.best_params_}")

GS_poly = GS
r2_poly = GS_poly.fit(X_train_poly, y_train).score(X_test_poly, y_test)
print(f"R2 (con): {r2_poly:.1%} con {GS_poly.best_params_}")

print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 3.4 AdaBoost: `AdaBoostRegressor`, `estimator=None`, `n_estimators=50`

Salida esperada:

```text
R2 (sin): 85.2% con {'estimator__max_depth': 5, 'n_estimators': 20}
R2 (con): 89.2% con {'estimator__max_depth': 5, 'n_estimators': 20}
Tiempo (hh:mm:ss): 00:00:18
```

In [ ]:
import time
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

rgs = AdaBoostRegressor(estimator=DecisionTreeRegressor(), random_state=23)
G = {"estimator__max_depth": [2, 5, 10, 20, 50, None], "n_estimators": [10, 20, 50, 100, 200, 500]}
splitter = ShuffleSplit(n_splits=10, test_size=0.1, random_state=23)
GS = GridSearchCV(rgs, G, scoring='r2', n_jobs=-1, cv=splitter, pre_dispatch=8)
r2 = GS.fit(X_train, y_train).score(X_test, y_test)
print(f"R2 (sin): {r2:.1%} con {GS.best_params_}")

GS_poly = GS
r2_poly = GS_poly.fit(X_train_poly, y_train).score(X_test_poly, y_test)
print(f"R2 (con): {r2_poly:.1%} con {GS_poly.best_params_}")

print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 3.5 Gradient boosting: `GradientBoostingRegressor`, `n_estimators=100`, `max_depth=3`

Salida esperada:

```text
R2 (sin): 86.3% con {'max_depth': 2, 'n_estimators': 100}
R2 (con): 88.6% con {'max_depth': 2, 'n_estimators': 100}
Tiempo (hh:mm:ss): 00:00:13
```

In [ ]:
import time
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

rgs = GradientBoostingRegressor(random_state=23)
G = {"n_estimators": [10, 20, 50, 100, 200, 500], "max_depth": [2, 5, 10, 20, 50, None]}
splitter = ShuffleSplit(n_splits=10, test_size=0.1, random_state=23)
GS = GridSearchCV(rgs, G, scoring='r2', n_jobs=-1, cv=splitter, pre_dispatch=8)
r2 = GS.fit(X_train, y_train).score(X_test, y_test)
print(f"R2 (sin): {r2:.1%} con {GS.best_params_}")

GS_poly = GS
r2_poly = GS_poly.fit(X_train_poly, y_train).score(X_test_poly, y_test)
print(f"R2 (con): {r2_poly:.1%} con {GS_poly.best_params_}")

print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 3.6 Gradient boosting para datasets grandes (10K+ datos): `HistGradientBoostingRegressor`, `max_depth=None`

Salida esperada:

```text
R2 (sin): 87.6% con {'max_depth': 10}
R2 (con): 87.0% con {'max_depth': 10}
Tiempo (hh:mm:ss): 00:00:07
```

In [ ]:
import time
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

rgs = HistGradientBoostingRegressor(random_state=23)
G = {"max_depth": [2, 5, 10, 20, 50, None]}
splitter = ShuffleSplit(n_splits=10, test_size=0.1, random_state=23)
GS = GridSearchCV(rgs, G, scoring='r2', n_jobs=-1, cv=splitter, pre_dispatch=8)
r2 = GS.fit(X_train, y_train).score(X_test, y_test)
print(f"R2 (sin): {r2:.1%} con {GS.best_params_}")

GS_poly = GS
r2_poly = GS_poly.fit(X_train_poly, y_train).score(X_test_poly, y_test)
print(f"R2 (con): {r2_poly:.1%} con {GS_poly.best_params_}")

print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 3.7 Gradient boosting con XGBoost: `XGBRegressor`, `max_depth=None`

Salida esperada:

```text
R2 (sin): 86.8% con {'max_depth': 2}
R2 (con): 87.7% con {'max_depth': 2}
Tiempo (hh:mm:ss): 00:00:04
```

In [ ]:
import time
from xgboost import XGBRegressor
from sklearn.model_selection import ShuffleSplit, GridSearchCV

start = time.time()

rgs = XGBRegressor(random_state=23)
G = {"max_depth": [2, 5, 10, 20, 50, None]}
splitter = ShuffleSplit(n_splits=10, test_size=0.1, random_state=23)
GS = GridSearchCV(rgs, G, scoring='r2', n_jobs=-1, cv=splitter, pre_dispatch=8)
r2 = GS.fit(X_train, y_train).score(X_test, y_test)
print(f"R2 (sin): {r2:.1%} con {GS.best_params_}")

GS_poly = GS
r2_poly = GS_poly.fit(X_train_poly, y_train).score(X_test_poly, y_test)
print(f"R2 (con): {r2_poly:.1%} con {GS_poly.best_params_}")

print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

---
## 4. Ejercicio de clasificación

**Ejercicio:** aplica (algunas de) las técnicas vistas a Fashion-MNIST con $K^{(\text{pca})} = 600$ y a CIFAR-10 con $K^{(\text{pca})} = 1000$, y trata de alcanzar (y mejorar) los resultados de referencia dados a continuación.

**Precisión (%) en test:** solo HistGradBoost y XGBoost obtienen resultados competitivos.

|                   | MNIST<br>$K=400$ | Fashion<br>$K=600$ | Fashion<br>$K=400$ | CIFAR-10<br>$K=1000$ | CIFAR-10<br>$K=400$ |
|-------------------|:----:|:----:|:----:|:----:|:----:|
| **LR**            | 92.6 | 84.4 | −    | 41.1 | −    |
| **LDA**           | 87.3 | 81.6 | −    | 40.0 | −    |
| **QDA**           | 95.8 | 76.6 | −    | 53.0 | −    |
| **QLR**           | 98.3 | −    | 89.4 | −    | 52.6 |
| **KNN**           | 97.0 | 85.8 | −    | 35.6 | −    |
| **CART**          | 83.6 | 74.8 | −    | 30.8 | −    |
| **Bagging**       | 87.8 | 79.5 | −    | 28.8 | −    |
| **Bosques**       | 93.6 | 84.2 | −    | 38.9 | −    |
| **AdaBoost**      | 84.0 | 75.9 | −    | 18.3 | −    |
| **GradBoost**     | 72.0 | 72.7 | −    | 29.4 | −    |
| **HistGradBoost** | 96.3 | 87.0 | −    | 49.9 | −    |
| **XGBoost**       | 95.6 | 86.3 | −    | 49.3 | −    |

### 4.1 Fashion-MNIST ($K = 600$)

#### 4.1.0 Lectura y reducción de la dimensión

Salida esperada: `(60000, 600) (60000,) (10000, 600) (10000,)`

In [ ]:
# Tu código aquí

#### 4.1.1 CART: `DecisionTreeClassifier`

Salida esperada:

```text
Precisión: 74.8% con {'max_depth': 10}
Tiempo (hh:mm:ss): 00:01:23
```

In [ ]:
# Tu código aquí

#### 4.1.2 Bagging: `BaggingClassifier`

Salida esperada:

```text
Precisión: 79.5% con {'estimator__max_depth': 20, 'n_estimators': 5}
Tiempo (hh:mm:ss): 00:03:49
```

In [ ]:
# Tu código aquí

#### 4.1.3 Bosques: `RandomForestClassifier`

Salida esperada:

```text
Precisión: 84.2% con {'max_depth': 20, 'n_estimators': 100}
Tiempo (hh:mm:ss): 00:02:58
```

In [ ]:
# Tu código aquí

#### 4.1.4 AdaBoost: `AdaBoostClassifier`

Salida esperada:

```text
Precisión: 75.9% con {'estimator__max_depth': 20, 'n_estimators': 3}
Tiempo (hh:mm:ss): 00:03:39
```

In [ ]:
# Tu código aquí

#### 4.1.5 Gradient boosting: `GradientBoostingClassifier`

Salida esperada:

```text
Precisión: 72.7% con {'max_depth': 3, 'n_estimators': 2}
Tiempo (hh:mm:ss): 00:04:45
```

In [ ]:
# Tu código aquí

#### 4.1.6 Gradient boosting para datasets grandes (10K+ datos): `HistGradientBoostingClassifier`

Salida esperada:

```text
Precisión: 87.0% con {'max_depth': 20}
Tiempo (hh:mm:ss): 00:03:41
```

In [ ]:
# Tu código aquí

#### 4.1.7 Gradient boosting con XGBoost: `XGBClassifier`

Salida esperada:

```text
Precisión: 86.3% con {'max_depth': 4}
Tiempo (hh:mm:ss): 00:03:47
```

In [ ]:
# Tu código aquí

### 4.2 CIFAR-10 ($K = 1000$)

#### 4.2.0 Lectura y reducción de la dimensión

Salida esperada: `(50000, 1000) (50000,) (10000, 1000) (10000,)`

In [ ]:
# Tu código aquí

#### 4.2.1 CART: `DecisionTreeClassifier`

Salida esperada:

```text
Precisión: 30.8% con {'max_depth': 10}
Tiempo (hh:mm:ss): 00:01:27
```

In [ ]:
# Tu código aquí

#### 4.2.2 Bagging: `BaggingClassifier`

Salida esperada:

```text
Precisión: 28.8% con {'estimator__max_depth': 20, 'n_estimators': 5}
Tiempo (hh:mm:ss): 00:04:26
```

In [ ]:
# Tu código aquí

#### 4.2.3 Bosques: `RandomForestClassifier`

Salida esperada:

```text
Precisión: 38.9% con {'max_depth': 20, 'n_estimators': 100}
Tiempo (hh:mm:ss): 00:03:15
```

In [ ]:
# Tu código aquí

#### 4.2.4 AdaBoost: `AdaBoostClassifier`

Salida esperada:

```text
Precisión: 18.3% con {'estimator__max_depth': 20, 'n_estimators': 3}
Tiempo (hh:mm:ss): 00:04:40
```

In [ ]:
# Tu código aquí

#### 4.2.5 Gradient boosting: `GradientBoostingClassifier`

Salida esperada:

```text
Precisión: 29.4% con {'max_depth': 3, 'n_estimators': 2}
Tiempo (hh:mm:ss): 00:06:24
```

In [ ]:
# Tu código aquí

#### 4.2.6 Gradient boosting para datasets grandes (10K+ datos): `HistGradientBoostingClassifier`

Salida esperada:

```text
Precisión: 49.9% con {'max_depth': 20}
Tiempo (hh:mm:ss): 00:05:19
```

In [ ]:
# Tu código aquí

#### 4.2.7 Gradient boosting con XGBoost: `XGBClassifier`

Salida esperada:

```text
Precisión: 49.3% con {'max_depth': 4}
Tiempo (hh:mm:ss): 00:02:38
```

In [ ]:
# Tu código aquí

---
## 5. Ejercicio de regresión

**Ejercicio:** aplica (algunas de) las técnicas vistas a Fish y California housing, sin y con ingeniería de características, y trata de alcanzar (y mejorar) los resultados de referencia dados a continuación.

**$R^2$ (%) en test:** en general, (gradient) boosting alcanza los mejores resultados.

|                   | auto-mpg<br>sin | auto-mpg<br>con | Fish<br>sin | Fish<br>con | housing<br>sin | housing<br>con |
|-------------------|:----:|:----:|:----:|:----:|:----:|:----:|
| **Lineal**        | 79.8 | 85.3 | 94.8 | 96.2 | 59.1 | 65.6 |
| **Kernel**        | 79.1 | 88.1 | 91.9 | 96.4 | 42.7 | 60.0 |
| **CART**          | 81.7 | 81.0 | 95.3 | 92.0 | 69.0 | 66.7 |
| **Bagging**       | 88.0 | 88.5 | 97.5 | 95.6 | 80.7 | 80.0 |
| **Bosques**       | 88.2 | 88.4 | 97.5 | 95.8 | 80.7 | 80.0 |
| **AdaBoost**      | 85.2 | 89.2 | 96.1 | 95.0 | 81.9 | 81.9 |
| **GradBoost**     | 86.3 | 88.6 | 97.9 | 94.6 | 82.2 | 82.1 |
| **HistGradBoost** | 87.6 | 87.0 | 96.9 | 96.8 | 83.7 | 83.4 |
| **XGBoost**       | 86.8 | 87.7 | 96.8 | 95.1 | 83.5 | 82.6 |

### 5.1 Fish

#### 5.1.0 Carga, preproceso, partición, estandarización y características polinómicas

In [ ]:
# Tu código aquí

#### 5.1.1 CART: `DecisionTreeRegressor`

Salida esperada:

```text
R2 (sin): 95.3% con {'max_depth': 10}
R2 (con): 92.0% con {'max_depth': 10}
Tiempo (hh:mm:ss): 00:00:02
```

In [ ]:
# Tu código aquí

#### 5.1.2 Bagging: `BaggingRegressor`

Salida esperada:

```text
R2 (sin): 97.5% con {'estimator__max_depth': 20, 'n_estimators': 200}
R2 (con): 95.6% con {'estimator__max_depth': 20, 'n_estimators': 200}
Tiempo (hh:mm:ss): 00:00:12
```

In [ ]:
# Tu código aquí

#### 5.1.3 Bosques: `RandomForestRegressor`

Salida esperada:

```text
R2 (sin): 97.5% con {'max_depth': 20, 'n_estimators': 200}
R2 (con): 95.8% con {'max_depth': 20, 'n_estimators': 200}
Tiempo (hh:mm:ss): 00:00:09
```

In [ ]:
# Tu código aquí

#### 5.1.4 AdaBoost: `AdaBoostRegressor`

Salida esperada:

```text
R2 (sin): 96.1% con {'estimator__max_depth': 10, 'n_estimators': 200}
R2 (con): 95.0% con {'estimator__max_depth': 10, 'n_estimators': 200}
Tiempo (hh:mm:ss): 00:00:11
```

In [ ]:
# Tu código aquí

#### 5.1.5 Gradient boosting: `GradientBoostingRegressor`

Salida esperada:

```text
R2 (sin): 97.9% con {'max_depth': 5, 'n_estimators': 50}
R2 (con): 94.6% con {'max_depth': 5, 'n_estimators': 50}
Tiempo (hh:mm:ss): 00:00:08
```

In [ ]:
# Tu código aquí

#### 5.1.6 Gradient boosting para datasets grandes (10K+ datos): `HistGradientBoostingRegressor`

Salida esperada:

```text
R2 (sin): 96.9% con {'max_depth': 2}
R2 (con): 96.8% con {'max_depth': 2}
Tiempo (hh:mm:ss): 00:00:04
```

In [ ]:
# Tu código aquí

#### 5.1.7 Gradient boosting con XGBoost: `XGBRegressor`

Salida esperada:

```text
R2 (sin): 96.8% con {'max_depth': 5}
R2 (con): 95.1% con {'max_depth': 5}
Tiempo (hh:mm:ss): 00:00:04
```

In [ ]:
# Tu código aquí

### 5.2 California housing

#### 5.2.0 Carga, preproceso, partición, estandarización y características polinómicas

In [ ]:
# Tu código aquí

#### 5.2.1 CART: `DecisionTreeRegressor`

Salida esperada:

```text
R2 (sin): 69.0% con {'max_depth': 10}
R2 (con): 66.7% con {'max_depth': 10}
Tiempo (hh:mm:ss): 00:00:05
```

In [ ]:
# Tu código aquí

#### 5.2.2 Bagging: `BaggingRegressor`

Salida esperada:

```text
R2 (sin): 80.7% con {'estimator__max_depth': 50, 'n_estimators': 50}
R2 (con): 80.0% con {'estimator__max_depth': 50, 'n_estimators': 50}
Tiempo (hh:mm:ss): 00:02:31
```

In [ ]:
# Tu código aquí

#### 5.2.3 Bosques: `RandomForestRegressor`

Salida esperada:

```text
R2 (sin): 80.7% con {'max_depth': 50, 'n_estimators': 50}
R2 (con): 80.0% con {'max_depth': 50, 'n_estimators': 50}
Tiempo (hh:mm:ss): 00:02:30
```

In [ ]:
# Tu código aquí

#### 5.2.4 AdaBoost: `AdaBoostRegressor`

Salida esperada:

```text
R2 (sin): 81.9% con {'estimator__max_depth': 50, 'n_estimators': 50}
R2 (con): 81.9% con {'estimator__max_depth': 50, 'n_estimators': 50}
Tiempo (hh:mm:ss): 00:03:18
```

In [ ]:
# Tu código aquí

#### 5.2.5 Gradient boosting: `GradientBoostingRegressor`

Salida esperada:

```text
R2 (sin): 82.2% con {'max_depth': 10, 'n_estimators': 50}
R2 (con): 82.1% con {'max_depth': 10, 'n_estimators': 50}
Tiempo (hh:mm:ss): 00:03:58
```

In [ ]:
# Tu código aquí

#### 5.2.6 Gradient boosting para datasets grandes (10K+ datos): `HistGradientBoostingRegressor`

Salida esperada:

```text
R2 (sin): 83.7% con {'max_depth': 10}
R2 (con): 83.4% con {'max_depth': 10}
Tiempo (hh:mm:ss): 00:00:03
```

In [ ]:
# Tu código aquí

#### 5.2.7 Gradient boosting con XGBoost: `XGBRegressor`

Salida esperada:

```text
R2 (sin): 83.5% con {'max_depth': 5}
R2 (con): 82.6% con {'max_depth': 5}
Tiempo (hh:mm:ss): 00:01:25
```

In [ ]:
# Tu código aquí